In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, AdamW, SGD
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.datasets import load_breast_cancer, load_iris, load_diabetes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img)
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)


In [ ]:
# 2. Create TensorDataset objects

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)



In [ ]:
# 3. Create DataLoaders

from torch.utils.data import DataLoader

# DataLoader for training data
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)
# DataLoader for test/validation data
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [ ]:
# 4. Print shape of one batch

# Get the first batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")




In [ ]:
images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i].squeeze().view(), cmap='gray')
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()



In [ ]:
# Task 1: Write your model class here:


class MulticlassNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.layer4 = nn.Linear(hidden_dim, num_classes)  # one per class!
        self.relu = nn.ReLU()
        # NO softmax! CrossEntropyLoss handles it!

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.relu(self.layer3(x))

        x = self.layer4(x)  # raw logits
        return x

In [ ]:
# Task 2: Write your training loop here:
INPUT_DIM = X_train.shape[1]
HIDDEN_DIM = 64
NUM_CLASSES = 10 # for multiclass
LEARNING_RATE = 0.001
NUM_EPOCHS = 20


def train_one_epoch(model, optimizer, criterion, train_loader, device, task):
    model.train()  # TRAINING MODE!
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.view(X_batch.size(0), -1).to(device) # flatten image to a vector
        y_batch = y_batch.to(device)


        # Forward
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        # Backward - THIS ORDER MATTERS!
        optimizer.zero_grad()  # 1️⃣ Clear gradients
        loss.backward()        # 2️⃣ Compute gradients
        optimizer.step()       # 3️⃣ Update weights

        running_loss += loss.item()

    return running_loss / len(train_loader)

In [ ]:
# Task 3: Write your validation loop here:

def validate(model, criterion, test_loader, device, task):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # NO GRADIENTS!
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            y_batch_loss = y_batch

            i_, predicted = torch.max(outputs, dim=1)
            correct += (predicted == y_batch).sum().item()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch_loss)
            running_loss += loss.item()


    avg_loss = running_loss / len(test_loader)
    accuracy = correct / total if task != "regression" else 0
    return avg_loss, accuracy

In [ ]:
rain_losses = []
val_losses = []
val_accs = []

print("Training started...")

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device, TASK)
    val_loss, val_acc = validate(model, criterion, test_loader, device, TASK)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:3d}/{NUM_EPOCHS}] "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f}")

print(f"Done! Final Accuracy: {val_accs[-1]:.4f}")

In [ ]:
# Task 4: Define device, model, loss, optimizer:

TASK = "multiclass"


INPUT_DIM = 3*36*36
HIDDEN_DIM = 14
OUtPUT_DIM = 1
LEARNING_RATE = 0.001
NUM_EPOCHS = 20

criterion = nn.CrossEntropyLoss()
model = MulticlassNet(INPUT_DIM, HIDDEN_DIM, NUM_CLASSES).to(device)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

train_losses = []
val_losses = []
val_accs = []



In [ ]:
# Task 5: Start training for 20 epochs:
# Run Training
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(NUM_EPOCHS):
  # Train one epoch
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

  # Validate
  val_loss = validate(model, criterion, test_loader, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  if (epoch + 1) % 5 == 0:
    print(
      f'Epoch [{epoch+1}/{num_epochs}], '
      f'Train Loss: {train_loss:.4f}, '
      f'Val Loss: {val_loss:.4f}'
    )



In [ ]:
# Task 1: Write your code here:

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curves')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_accs, label='Val Accuracy', color='green')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: